# Stage 10B: Time-Aware Classification Baseline

**Project context:** SPY next-day high-volatility risk alert. This notebook implements additional lag/rolling features, a chronological split, and a reproducible `sklearn` pipeline.

The target is positive when next-day absolute return exceeds the **training-period** 90th percentile. All features at date `t` are available after that date closes; no random split or full-sample threshold is used.


## 1. Reproducible setup


In [1]:
from pathlib import Path
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "homework" / "homework10b").is_dir():
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "homework" / "homework10b").is_dir():
            ROOT = candidate
            break
HOMEWORK = ROOT / "homework" / "homework10b"
RAW = HOMEWORK / "data" / "raw"
PROCESSED = HOMEWORK / "data" / "processed"
REPORTS = HOMEWORK / "reports"
for directory in [PROCESSED, REPORTS]:
    directory.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT)


Repository root: /Users/cengchengyu/Documents/NYU/Class/Boot Camp 4/CS HW/bootcamp_chengyu_zeng


## 2. Leakage-safe lag, rolling, and momentum features


In [2]:
snapshot_path = RAW / "spy_feature_candidates_stage09_snapshot.csv"
frame = pd.read_csv(snapshot_path, parse_dates=["date"]).sort_values("date", kind="stable").reset_index(drop=True)

# Every field below ends at date t; the only forward shift is the outcome already present in the snapshot.
frame["return_lag_1_t"] = frame["return_t"].shift(1)
frame["rolling_abs_return_mean_5_t"] = frame["abs_return_t"].rolling(5).mean()
frame["rolling_return_std_10_t"] = frame["return_t"].rolling(10).std(ddof=1)
frame["momentum_5_t"] = frame["close"].pct_change(5, fill_method=None)

feature_columns = [
    "abs_return_t", "intraday_range_t", "log_volume_change_t", "rolling_volatility_5_t",
    "return_lag_1_t", "rolling_abs_return_mean_5_t", "rolling_return_std_10_t", "momentum_5_t",
    "weekday_Monday", "weekday_Tuesday", "weekday_Wednesday", "weekday_Thursday", "weekday_Friday",
]
target_column = "next_day_abs_return"
model_frame = frame.loc[:, ["date", *feature_columns, target_column]].dropna().reset_index(drop=True)

required_snapshot_columns = {
    "date", "close", "return_t", "abs_return_t", "intraday_range_t",
    "log_volume_change_t", "rolling_volatility_5_t", "next_day_abs_return",
}
assert len(frame) == 2512 and required_snapshot_columns.issubset(frame.columns)
assert model_frame["date"].is_monotonic_increasing and model_frame["date"].is_unique
assert model_frame["date"].max() < frame["date"].max()  # terminal outcome is unavailable by construction
assert len(feature_columns) >= 2
print("Model-ready observations:", len(model_frame))


Model-ready observations: 2501


## 3. Chronological split and train-only event definition


In [3]:
split_index = int(len(model_frame) * 0.80)
train = model_frame.iloc[:split_index].copy()
test = model_frame.iloc[split_index:].copy()

high_volatility_threshold = train[target_column].quantile(0.90)
train["high_volatility_next_day"] = (train[target_column] > high_volatility_threshold).astype("int8")
test["high_volatility_next_day"] = (test[target_column] > high_volatility_threshold).astype("int8")

assert train["date"].max() < test["date"].min()
assert train["high_volatility_next_day"].mean() == 0.10
assert high_volatility_threshold > 0
print("Train rows:", len(train), "Test rows:", len(test))
print("Training-only 90th-percentile threshold:", f"{high_volatility_threshold:.4%}")
print("Positive rate — train:", f"{train['high_volatility_next_day'].mean():.2%}", "test:", f"{test['high_volatility_next_day'].mean():.2%}")


Train rows: 2000 Test rows: 501
Training-only 90th-percentile threshold: 1.6454%
Positive rate — train: 10.00% test: 8.18%


## 4. Pipeline, probability, and fixed homework cutoff


In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
pipeline.fit(train[feature_columns], train["high_volatility_next_day"])

probability = pipeline.predict_proba(test[feature_columns])[:, 1]
alert_cutoff = 0.50  # Project work will select this operating point on a validation period.
prediction = (probability >= alert_cutoff).astype("int8")
y_test = test["high_volatility_next_day"].to_numpy()

metrics = pd.DataFrame([{
    "train_rows": len(train),
    "test_rows": len(test),
    "training_quantile_90": high_volatility_threshold,
    "alert_cutoff": alert_cutoff,
    "test_positive_rate": y_test.mean(),
    "alert_rate": prediction.mean(),
    "accuracy": accuracy_score(y_test, prediction),
    "precision": precision_score(y_test, prediction, zero_division=0),
    "recall": recall_score(y_test, prediction, zero_division=0),
    "f1": f1_score(y_test, prediction, zero_division=0),
    "pr_auc": average_precision_score(y_test, probability),
}])
metrics_path = PROCESSED / "classification_metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics.T.rename(columns={0: "value"}))


,value
train_rows,2000.000000
test_rows,501.000000
training_quantile_90,0.016454
alert_cutoff,0.500000
test_positive_rate,0.081836
alert_rate,0.177645
accuracy,0.828343
precision,0.247191
recall,0.536585
f1,0.338462


## 5. Confusion matrix and future-period probability diagnostics


In [5]:
matrix = confusion_matrix(y_test, prediction)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(matrix, display_labels=["normal", "high volatility"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Future-period confusion matrix")

axes[1].plot(test["date"], probability, color="#2563EB", linewidth=1.1, label="predicted probability")
axes[1].axhline(alert_cutoff, color="#DC2626", linestyle="--", label="homework cutoff")
axes[1].scatter(
    test.loc[y_test == 1, "date"], probability[y_test == 1],
    color="#F59E0B", s=18, label="actual high-volatility day", zorder=3,
)
axes[1].set(title="Future-period risk probabilities", xlabel="Date", ylabel="Probability")
axes[1].legend(loc="upper right")
fig.autofmt_xdate()
fig.tight_layout()
diagnostics_path = REPORTS / "classification_diagnostics.png"
fig.savefig(diagnostics_path, dpi=150, bbox_inches="tight")
plt.show()


/var/folders/hy/9nh5rd9526vd43l1zms4t1lh0000gn/T/ipykernel_77001/1725704344.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Interpretation and risk-aware conclusion

- **What works:** the pipeline is chronological and reproducible; train-only target construction prevents a direct full-sample threshold leak. The probability ranking is evaluated with PR-AUC, not accuracy alone.
- **What fails or remains limited:** the positive class is rare, so a balanced logistic model can raise recall while producing false positives. A 0.50 cutoff is arbitrary and is not a final operating decision.
- **Assumptions and risks:** logistic regression assumes a stable conditional relationship and can fail after a volatility regime shift. Rolling features can react too slowly, and target/feature timing must be rechecked whenever the decision window changes.
- **Conclusion:** this is a useful transparent baseline, not a deployment-ready alert. The project must use a separate chronological validation period to choose model and alert cutoff, then reserve its final future test period for one-time evaluation.


In [6]:
assert metrics.shape == (1, 11)
assert np.isfinite(metrics.to_numpy(dtype=float)).all()
assert 0 < metrics.loc[0, "test_positive_rate"] < 1
assert 0 < metrics.loc[0, "pr_auc"] <= 1
assert diagnostics_path.is_file() and diagnostics_path.stat().st_size > 0
assert metrics_path.is_file() and metrics_path.stat().st_size > 0
print("Stage10B checks passed.")
print("Metrics:", metrics_path.name)
print("Diagnostics:", diagnostics_path.name)


Stage10B checks passed.
Metrics: classification_metrics.csv
Diagnostics: classification_diagnostics.png
